In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from fairlearn.metrics import (MetricFrame, demographic_parity_difference, equalized_odds_difference)
from fairlearn.postprocessing import ThresholdOptimizer

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

SEED = 42
np.random.seed(SEED)

TARGET = "hospital_expire_flag"
protectedAttrs = ["gender", "race_group", "insurance_group", "age_group"]
mainAttr = "race_group"

print("IMports worked")

In [ ]:
# This loads the data:
cohort = pd.read_csv("cohort.csv")

print("Top missings:")
print((cohort.isna().mean() * 100).sort_values(ascending=False).head(10))
print(f"Shape: {cohort.shape}")

In [ ]:
# Dropping useless colums like ID and the other ones mentioned below
cohort = cohort.drop(columns=["subject_id", "hadm_id", "stay_id", "marital_status", "long_icu_stay", "los_hospital_hrs", "los", "dod_known", "lab_lactate", "lab_ph"])

# Finding the Medians for numeric columns
num_cols = cohort.select_dtypes(include="number").columns.tolist()
numeric_impute_cols = [c for c in num_cols if c != TARGET]
for col in numeric_impute_cols:
    cohort[col] = cohort[col].fillna(cohort[col].median())

# Age bins
cohort["age_group"] = pd.cut(
    cohort["anchor_age"],
    bins=[0, 40, 60, 200],
    labels=["Under 40", "40 to 60", "Over 60"]
)
cohort = cohort.drop(columns=["anchor_age"])

print(f"\nFinal shape: {cohort.shape}")
print(f"Mortality rate: {cohort[TARGET].mean():.2%}")

In [ ]:
protectedAttrs_df = cohort[protectedAttrs].copy()
cohort_encoded = pd.get_dummies(
    cohort,
    columns=["race_group", "insurance_group", "age_group"],
    drop_first=False
)

feature_cols = [c for c in cohort_encoded.columns if c != TARGET]
X = cohort_encoded[feature_cols]
y = cohort[TARGET]

print(f"X shape: {X.shape}")
print(f"Class balance — 0: {(y==0).sum()}  1: {(y==1).sum()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# proptected attributes aligned to splits by index
sens_test  = protectedAttrs_df.loc[X_test.index]
sens_train = protectedAttrs_df.loc[X_train.index]
s_test     = sens_test[mainAttr]
s_train    = sens_train[mainAttr]

# Scaling (used for LR)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols, index=X_test.index
)

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"Train mortality: {y_train.mean():.2%}  Test mortality: {y_test.mean():.2%}")

Model training



In [ ]:
#Logistic Regression
lr_base = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
lr_base.fit(X_train_scaled, y_train)
lr_pred = lr_base.predict(X_test_scaled)
lr_prob = lr_base.predict_proba(X_test_scaled)[:, 1]

#R andom Forest
rf_base = RandomForestClassifier(
    n_estimators=100, random_state=SEED,
    class_weight="balanced_subsample", min_samples_leaf=5
)
rf_base.fit(X_train, y_train)
rf_pred = rf_base.predict(X_test)
rf_prob = rf_base.predict_proba(X_test)[:, 1]

# XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
xgb_base = XGBClassifier(
    n_estimators=100, random_state=SEED,
    eval_metric="logloss", verbosity=0,
    scale_pos_weight=neg / pos
)
xgb_base.fit(X_train, y_train)
xgb_pred = xgb_base.predict(X_test)
xgb_prob = xgb_base.predict_proba(X_test)[:, 1]

print("baseline model is now trainged")

Additional functions to help with this 


In [ ]:
def get_standard_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "roc_auc":round(roc_auc_score(y_true, y_prob), 4),
        "fpr": round(fp / (fp + tn) if (fp + tn) > 0 else 0, 4),
        "fnr": round(fn / (fn + tp) if (fn + tp) > 0 else 0, 4),
    }


def get_fairness_metrics(y_true, y_pred, sensitive):
    dpd = demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive)
    eod = equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive)
    mf = MetricFrame(
        metrics={"tpr": lambda yt, yp: recall_score(yt, yp, zero_division=0)},
        y_true=y_true, y_pred=y_pred,
        sensitive_features=sensitive
    )
    tprs = mf.by_group["tpr"]
    equal_opp = float(tprs.max() - tprs.min())
    return {
        "dem_parity_diff": round(abs(dpd), 4),
        "equal_opp_diff":  round(equal_opp, 4),
        "equalized_odds":  round(abs(eod), 4),
    }


def evaluate_model(method, model_name, y_true, y_pred, y_prob, sensitive):
    row = {"method": method, "model": model_name}
    row.update(get_standard_metrics(y_true, y_pred, y_prob))
    row.update(get_fairness_metrics(y_true, y_pred, sensitive))
    return row

Evaluation of the Baseline Model:



In [ ]:
all_results = []

for model_name, y_pred, y_prob in [
    ("LR",  lr_pred,  lr_prob),
    ("RF",  rf_pred,  rf_prob),
    ("XGB", xgb_pred, xgb_prob),
]:
    row = evaluate_model("Baseline", model_name, y_test, y_pred, y_prob, s_test)
    all_results.append(row)
    print(f"\nBaseline {model_name}")
    for k, v in row.items():
        if k not in ("method", "model"):
            print(f"  {k:25s}: {v}")

## 6. Group-Level Analysis (FPR / FNR / AUC per demographic)

In [ ]:
# Build test_df for  all predictions + demographics in one table
test_df = sens_test.copy()
test_df["y_true"] = y_test.values
test_df["lr_pred"] = lr_pred
test_df["lr_prob"] = lr_prob
test_df["rf_pred"] = rf_pred
test_df["rf_prob"] = rf_prob
test_df["xgb_pred"] = xgb_pred
test_df["xgb_prob"] = xgb_prob

print(f"test_df shape: {test_df.shape}")
test_df.head(5)

In [ ]:
def plot_group_metrics(attr, filename):
    groups = sorted(test_df[attr].dropna().astype(str).unique())
    models_info = [
        ("LR",  "lr_pred",  "lr_prob",  "steelblue"),
        ("RF",  "rf_pred",  "rf_prob",  "tomato"),
        ("XGB", "xgb_pred", "xgb_prob", "seagreen"),
    ]
    metrics = ["FPR", "FNR", "AUC"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"{attr.replace('_', ' ').title()} — Group Metrics", fontweight="bold", fontsize=13)
    bar_width = 0.25
    x = np.arange(len(groups))

    for ax, metric in zip(axes, metrics):
        for i, (label, pred_col, prob_col, colour) in enumerate(models_info):
            vals = []
            for g in groups:
                gdf = test_df[test_df[attr].astype(str) == g]
                yt = gdf["y_true"].values
                yp = gdf[pred_col].values
                yb = gdf[prob_col].values
                if len(np.unique(yt)) < 2:
                    vals.append(np.nan)
                    continue
                tn_, fp_, fn_, tp_ = confusion_matrix(yt, yp, labels=[0,1]).ravel()
                if metric == "FPR":
                    vals.append(fp_ / (fp_ + tn_) if (fp_ + tn_) > 0 else 0)
                elif metric == "FNR":
                    vals.append(fn_ / (fn_ + tp_) if (fn_ + tp_) > 0 else 0)
                else:
                    vals.append(roc_auc_score(yt, yb))
            ax.bar(x + i * bar_width, vals, bar_width, label=label, color=colour)

        ax.set_title(metric, fontsize=11)
        ax.set_xticks(x + bar_width)
        ax.set_xticklabels(groups, rotation=30, ha="right")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {filename}")

for attr in protectedAttrs:
    plot_group_metrics(attr, f"figures/group_metrics_{attr}.png")

Mitigation Method 1 - Reweighing (Pre-processing)

Assigns higher sample weights to underrepresented group * outcome combinations so the model learns more from rare cases (e.g. minority patients who died).

In [ ]:
# Build combined race * outcome label for weighting
group_outcome_train = s_train.astype(str) + "_" + y_train.astype(str)
rw_weights = compute_sample_weight("balanced", group_outcome_train)

print("Sample weight summary:")
weight_df = pd.DataFrame({"group_outcome": group_outcome_train, "weight": rw_weights})
print(weight_df.groupby("group_outcome")["weight"].agg(["mean", "count"]).round(3))

In [ ]:
# LR reweighing
lr_rw = LogisticRegression(max_iter=1000, random_state=SEED)
lr_rw.fit(X_train_scaled, y_train, sample_weight=rw_weights)
lr_rw_pred = lr_rw.predict(X_test_scaled)
lr_rw_prob = lr_rw.predict_proba(X_test_scaled)[:, 1]

# RF reweighing
rf_rw = RandomForestClassifier(n_estimators=100, random_state=SEED, min_samples_leaf=5)
rf_rw.fit(X_train, y_train, sample_weight=rw_weights)
rf_rw_pred = rf_rw.predict(X_test)
rf_rw_prob = rf_rw.predict_proba(X_test)[:, 1]

# xgBoost reweighting
xgb_rw = XGBClassifier(n_estimators=100, random_state=SEED, eval_metric="logloss", verbosity=0)
xgb_rw.fit(X_train, y_train, sample_weight=rw_weights)
xgb_rw_pred = xgb_rw.predict(X_test)
xgb_rw_prob = xgb_rw.predict_proba(X_test)[:, 1]

for model_name, y_pred, y_prob in [
    ("LR",  lr_rw_pred,  lr_rw_prob),
    ("RF",  rf_rw_pred,  rf_rw_prob),
    ("XGB", xgb_rw_pred, xgb_rw_prob),
]:
    row = evaluate_model("Reweighing", model_name, y_test, y_pred, y_prob, s_test)
    all_results.append(row)
    print(f"\nReweighing {model_name}")
    for k, v in row.items():
        if k not in ("method", "model"):
            print(f"  {k:25s}: {v}")

Mitigation Method 2 - SMOTE Resampling (Pre - P{rocessing)

Generates synthetic minority-class samples to balance the training set. Applied only to training data but I havent' touched the test set.

In [ ]:
print(f"Before SMOTE: {X_train.shape[0]} samples | Class: {dict(y_train.value_counts())}")

smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Scale SMOTE data using the already-fitted scaler - not refitting.
X_train_sm_scaled = pd.DataFrame(
    scaler.transform(X_train_sm),
    columns=feature_cols
)

print(f"After SMOTE:  {X_train_sm.shape[0]} samples | Class: {dict(pd.Series(y_train_sm).value_counts())}")

In [ ]:
# LR SMOTE
lr_sm = LogisticRegression(max_iter=1000, random_state=SEED)
lr_sm.fit(X_train_sm_scaled, y_train_sm)
lr_sm_pred = lr_sm.predict(X_test_scaled)
lr_sm_prob = lr_sm.predict_proba(X_test_scaled)[:, 1]

# RF SMOTE
rf_sm = RandomForestClassifier(n_estimators=100, random_state=SEED, min_samples_leaf=5)
rf_sm.fit(X_train_sm, y_train_sm)
rf_sm_pred = rf_sm.predict(X_test)
rf_sm_prob = rf_sm.predict_proba(X_test)[:, 1]

# XGBoost SMOTE
xgb_sm = XGBClassifier(n_estimators=100, random_state=SEED, eval_metric="logloss", verbosity=0)
xgb_sm.fit(X_train_sm, y_train_sm)
xgb_sm_pred = xgb_sm.predict(X_test)
xgb_sm_prob = xgb_sm.predict_proba(X_test)[:, 1]

for model_name, y_pred, y_prob in [
    ("LR",  lr_sm_pred,  lr_sm_prob),
    ("RF",  rf_sm_pred,  rf_sm_prob),
    ("XGB", xgb_sm_pred, xgb_sm_prob),
]:
    row = evaluate_model("SMOTE", model_name, y_test, y_pred, y_prob, s_test)
    all_results.append(row)
    print(f"\n=== SMOTE {model_name} ===")
    for k, v in row.items():
        if k not in ("method", "model"):
            print(f"  {k:25s}: {v}")

Mitigation Method 3 - Threshold Optimisation (Post - processing)

Using a different decision threshold now per demographic group so that FPR and FNR are equalised across groups. The model weights are unchanged - only when it decides to predict positive shifts per group.

In [ ]:
# LR threshold optimisation
thresh_lr = ThresholdOptimizer(
    estimator=lr_base,
    constraints="equalized_odds",
    predict_method="predict_proba",
    objective="balanced_accuracy_score"
)
thresh_lr.fit(X_train_scaled, y_train, sensitive_features=s_train.values)
thresh_lr_pred = thresh_lr.predict(X_test_scaled, sensitive_features=s_test.values)
thresh_lr_prob = lr_base.predict_proba(X_test_scaled)[:, 1]


In [ ]:
#RF threshold optimisation
thresh_rf = ThresholdOptimizer(
    estimator=rf_base,
    constraints="equalized_odds",
    predict_method="predict_proba",
    objective="balanced_accuracy_score"
)
thresh_rf.fit(X_train, y_train, sensitive_features=s_train.values)
thresh_rf_pred = thresh_rf.predict(X_test, sensitive_features=s_test.values)
thresh_rf_prob = rf_base.predict_proba(X_test)[:, 1]

In [ ]:
# xgboost manual threshold optimisation
xgb_probs_all = xgb_base.predict_proba(X_test.astype(np.float64))[:, 1]

group_thresholds = {}
for group in s_test.unique():
    mask = (s_test.values == group)
    y_g = y_test.values[mask]
    p_g = xgb_probs_all[mask]
    best_thresh, best_score = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.05):
        preds = (p_g >= t).astype(int)
        if len(np.unique(y_g)) < 2:
            continue
        tn_, fp_, fn_, tp_ = confusion_matrix(y_g, preds, labels=[0,1]).ravel()
        tpr_ = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0
        tnr_ = tn_ / (tn_ + fp_) if (tn_ + fp_) > 0 else 0
        score = (tpr_ + tnr_) / 2
        if score > best_score:
            best_score  = score
            best_thresh = t
    group_thresholds[group] = best_thresh

print("Per group thresholds:", group_thresholds)

thresh_xgb_pred = np.zeros(len(y_test), dtype=int)
for group, thresh in group_thresholds.items():
    mask = (s_test.values == group)
    thresh_xgb_pred[mask] = (xgb_probs_all[mask] >= thresh).astype(int)
thresh_xgb_prob = xgb_probs_all

In [ ]:
for model_name, y_pred, y_prob in [
    ("LR", thresh_lr_pred, thresh_lr_prob),
    ("RF", thresh_rf_pred, thresh_rf_prob),
    ("XGB", thresh_xgb_pred, thresh_xgb_prob),
]:
    row = evaluate_model("ThresholdOpt", model_name, y_test, y_pred, y_prob, s_test)
    all_results.append(row)
    print(f"\nThresholdOpt {model_name}")
    for k, v in row.items():
        if k not in ("method", "model"):
            print(f"{k:25s}: {v}")

Comparison Table for all the things now:


In [ ]:
results_df = pd.DataFrame(all_results)
col_order = [
    "method", "model",
    "roc_auc", "accuracy", "precision", "recall", "f1",
    "fpr", "fnr",
    "dem_parity_diff", "equal_opp_diff", "equalized_odds"
]
results_df = results_df[col_order]
print(results_df.to_string(index=False))

Visualisations:

In [ ]:
# Fairness metrics comparsions

methods = list(results_df["method"].unique())
models = ["LR", "RF", "XGB"]
colours = ["steelblue", "tomato", "seagreen"]
bar_width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Fairness Metrics by Method and Model", fontweight="bold", fontsize=13)

fairness_metrics = ["dem_parity_diff", "equal_opp_diff", "equalized_odds"]
metric_labels = ["Demographic Parity Diff", "Equal Opportunity Diff", "Equalized Odds Diff"]

for ax, metric, label in zip(axes, fairness_metrics, metric_labels):
    x = np.arange(len(methods))
    for i, (model, colour) in enumerate(zip(models, colours)):
        vals = [results_df[(results_df["method"] == m) & (results_df["model"] == model)][metric].values[0] for m in methods]
        ax.bar(x + i * bar_width, vals, bar_width, label=model, color=colour)
    ax.set_title(label, fontsize=10)
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel("Difference (lower = fairer)")
    ax.legend(fontsize=8)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

plt.tight_layout()
plt.savefig("figures/fairness_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/fairness_comparison.png")

Visualisation for performance metrics comparison:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Performance Metrics by Method and Model", fontweight="bold", fontsize=13)

perf_metrics = ["roc_auc", "f1", "recall"]
perf_labels = ["ROC-AUC", "F1 Score", "Recall (Death)"]

for ax, metric, label in zip(axes, perf_metrics, perf_labels):
    x = np.arange(len(methods))
    for i, (model, colour) in enumerate(zip(models, colours)):
        vals = [results_df[(results_df["method"] == m) & (results_df["model"] == model)][metric].values[0] for m in methods]
        ax.bar(x + i * bar_width, vals, bar_width, label=model, color=colour)
    ax.set_title(label, fontsize=10)
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("figures/performance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/performance_comparison.png")

Fairness and Accuracy Trade-off Curve Distinction Level

Each point is one model * method combination Bottom-left is ideal as it would be both fair AND accurate.

In [ ]:
method_colours = {
    "Baseline": "steelblue",
    "Reweighing": "tomato",
    "SMOTE": "seagreen",
    "ThresholdOpt": "darkorchid",
}
model_markers = {"LR": "o", "RF": "s", "XGB": "^"}

fig, ax = plt.subplots(figsize=(9, 6))

for _, row in results_df.iterrows():
    ax.scatter(
        row["equalized_odds"],
        row["roc_auc"],
        color=method_colours[row["method"]],
        marker=model_markers[row["model"]],
        s=130, zorder=3,
        edgecolors="black", linewidths=0.5
    )
    ax.annotate(
        f"{row['method']}\n{row['model']}",
        (row["equalized_odds"], row["roc_auc"]),
        textcoords="offset points", xytext=(7, 4), fontsize=7
    )

legend_elements = (
    [mpatches.Patch(facecolor=c, label=m) for m, c in method_colours.items()] +
    [mlines.Line2D([0], [0], marker=mk, color="w", markerfacecolor="grey", markersize=9, label=mod) for mod, mk in model_markers.items()]
)
ax.legend(handles=legend_elements, fontsize=8, loc="lower left")
ax.set_xlabel("Equalized Odds Difference  (lower = fairer)", fontsize=11)
ax.set_ylabel("ROC-AUC  (higher = better)", fontsize=11)
ax.set_title("Fairness–Accuracy Trade-off\n(race group as sensitive attribute)", fontweight="bold", fontsize=12)
ax.grid(True, alpha=0.3)
ax.annotate("← Fairer", xy=(ax.get_xlim()[0] + 0.01, ax.get_ylim()[1] * 0.995), fontsize=8, color="grey", style="italic")

plt.tight_layout()
plt.savefig("figures/tradeoff_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/tradeoff_curve.png")

Storting the results to csv files:

In [ ]:
results_df.to_csv("results/all_results.csv", index=False)
results_df[results_df["method"] == "Baseline"].to_csv("results/baseline_metrics.csv", index=False)
results_df[["method", "model", "dem_parity_diff", "equal_opp_diff", "equalized_odds"]].to_csv("results/fairness_metrics.csv", index=False)
results_df[results_df["method"] != "Baseline"].to_csv("results/mitigated_metrics.csv", index=False)

Summaries Tables for all the models with the differet metrics

In [ ]:
summary = results_df[["method", "model", "roc_auc", "f1", "dem_parity_diff", "equalized_odds"]]
summary = summary.sort_values(["model", "method"])

print(f"{'Method':<15} {'Model':<6} {'AUC':>7} {'F1':>7} {'Dem.Parity':>12} {'Eq.Odds':>10}")
print()
for _, r in summary.iterrows():
    print(f"{r['method']:<15} {r['model']:<6} {r['roc_auc']:>7.4f} {r['f1']:>7.4f} "
          f"{r['dem_parity_diff']:>12.4f} {r['equalized_odds']:>10.4f}")
print()
print("Note: fairness metrics - lower means more fair (0 meams perfect parity)")